# NB_LIBRO_05 · Reevaluación del mAP de los detectores de pollos

**Libro:** *AgroVisión: del píxel a la decisión* · capítulo 8

**Por qué existe este notebook.** El NB08 informa un mAP50-95 de 0,400 para YOLO11s, mientras que el NB02 informa 0,756. La causa está en la métrica: `torchmetrics` toma su valor `map` de la primera estadística de `pycocotools`, que se calcula siempre con un máximo de **100 detecciones por imagen**, aunque `map_50` y `map_75` usen el máximo indicado (1 000). En el conjunto de validación, 231 de las 452 imágenes tienen más de 100 aves y solo el 49 % de las aves anotadas cabe en ese límite: un detector perfecto obtendría 0,495.

**Qué hace.** Repite la inferencia del NB08 con los mismos modelos y parámetros y calcula el mAP50-95 con 100, 300 y 1 000 detecciones por imagen. Además guarda el número de aves que cuenta cada detector en cada imagen.

**Usa (solo lectura):** `AgroUTMACH_PIX/01_DATASET_PIO/PIO_TRAIN_VAL/data` (validación) y los tres modelos de `AgroUTMACH_PIX/03_MODELOS`. No modifica nada de los proyectos.

**Produce:** `AGROVISION_LIBRO/03_VALORES/reevaluacion_mAP.json` y `conteo_detectores_por_imagen.csv`, y descarga ambos en un zip pequeño.

**Cómo ejecutarlo:** seleccione *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)* y luego *Ejecutar todo*. Tarda entre 10 y 15 minutos. Suba al chat el zip que se descarga al final (no hace falta el notebook empaquetador).

In [ ]:
%pip install -q pycocotools ultralytics==8.4.172 pyyaml

In [ ]:
import os, re, json, time, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))
ROOT = BASE / 'AgroUTMACH_PIX'
DATOS = ROOT / '01_DATASET_PIO' / 'PIO_TRAIN_VAL' / 'data'
MODELOS = {
    'YOLO11s': ROOT / '03_MODELOS' / 'AgroUTMACH_PIX_YOLO11s_PIO_best.pt',
    'YOLO26s': ROOT / '03_MODELOS' / 'AgroUTMACH_PIX_YOLO26s_PIO_best.pt',
    'Faster R-CNN': ROOT / '03_MODELOS' / 'AgroUTMACH_PIX_FasterRCNN_ResNet50FPN_best.pt',
}
DIR_VAL = BASE / 'AGROVISION_LIBRO' / '03_VALORES'
DIR_VAL.mkdir(parents=True, exist_ok=True)
LOCAL = Path(os.environ.get('AGROVISION_LOCAL', '/content/PIO_VAL_LOCAL'))

UMBRAL_CONTEO = 0.25          # confianza mínima para contar un ave (como en NB08)
LIMITES = [100, 300, 1000]    # detecciones máximas por imagen que se comparan
EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

for nombre, ruta in MODELOS.items():
    print(f"{nombre:13s} {'OK' if ruta.exists() else 'NO ENCONTRADO'}  {ruta.name}")
assert (DATOS / 'images' / 'val').exists(), f'No se encuentra el dataset en {DATOS}'

## 1. Copia local de la validación y anotaciones en formato COCO

In [ ]:
if LOCAL.exists():
    shutil.rmtree(LOCAL)
for sub in ('images', 'labels'):
    shutil.copytree(DATOS / sub / 'val', LOCAL / sub / 'val')
imagenes = sorted(p for p in (LOCAL / 'images' / 'val').iterdir() if p.suffix.lower() in EXT_IMG)
print('Imágenes de validación:', len(imagenes))

PATRON = re.compile(r'^([CP])-W(\d+)', re.IGNORECASE)
gt_imgs, gt_anns, filas_img = [], [], []
ann_id = 1
for idx, ruta in enumerate(imagenes):
    with Image.open(ruta) as im:
        ancho, alto = im.size
    gt_imgs.append({'id': idx, 'file_name': ruta.name, 'width': ancho, 'height': alto})
    etq = LOCAL / 'labels' / 'val' / (ruta.stem + '.txt')
    n_cajas = 0
    if etq.exists():
        for linea in etq.read_text(encoding='utf-8').splitlines():
            partes = linea.split()
            if len(partes) < 5:
                continue
            _, xc, yc, bw, bh = map(float, partes[:5])
            x1 = max(0.0, (xc - bw / 2) * ancho); y1 = max(0.0, (yc - bh / 2) * alto)
            x2 = min(float(ancho), (xc + bw / 2) * ancho); y2 = min(float(alto), (yc + bh / 2) * alto)
            if x2 > x1 and y2 > y1:
                gt_anns.append({'id': ann_id, 'image_id': idx, 'category_id': 1, 'iscrowd': 0,
                                'bbox': [x1, y1, x2 - x1, y2 - y1], 'area': (x2 - x1) * (y2 - y1)})
                ann_id += 1; n_cajas += 1
    hallado = PATRON.match(ruta.name)
    filas_img.append({'archivo': ruta.name, 'galpon': hallado.group(1).upper() if hallado else '?',
                      'semana': int(hallado.group(2)) if hallado else 0, 'anotadas': n_cajas})
tabla_img = pd.DataFrame(filas_img)
INDICE = {ruta.name: i for i, ruta in enumerate(imagenes)}
n_aves = tabla_img['anotadas']
print('Aves anotadas:', int(n_aves.sum()), '| media por imagen:', round(float(n_aves.mean()), 1), '| máximo:', int(n_aves.max()))
techo = {k: round(float(n_aves.clip(upper=k).sum() / n_aves.sum()), 4) for k in LIMITES}
print('Fracción de aves anotadas que cabe en cada límite de detecciones:', techo)
print('Imágenes con más de 100 aves:', int((n_aves > 100).sum()), '| con más de 300:', int((n_aves > 300).sum()))

## 2. Métrica

`evaluar_coco` usa `pycocotools` y lee la matriz de precisión completa, de modo que el mAP50-95 se obtiene por separado para cada límite de detecciones. La fila `map_torchmetrics` reproduce el valor que devolvía el NB08.

In [ ]:
import io, contextlib
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

GT = COCO()
GT.dataset = {'images': gt_imgs, 'annotations': gt_anns, 'categories': [{'id': 1, 'name': 'Pollo'}]}
with contextlib.redirect_stdout(io.StringIO()):
    GT.createIndex()


def evaluar_coco(detecciones):
    # detecciones: lista de dicts {'image_id', 'category_id': 1, 'bbox': [x, y, ancho, alto], 'score'}
    if len(detecciones) == 0:
        return {}
    with contextlib.redirect_stdout(io.StringIO()):
        dt = GT.loadRes(detecciones)
        ev = COCOeval(GT, dt, 'bbox')
        ev.params.maxDets = LIMITES
        ev.evaluate(); ev.accumulate(); ev.summarize()
    prec = ev.eval['precision']          # [IoU, recall, clase, área, límite]
    rec = ev.eval['recall']              # [IoU, clase, área, límite]
    ious = ev.params.iouThrs
    i50 = int(np.argmin(np.abs(ious - 0.50))); i75 = int(np.argmin(np.abs(ious - 0.75)))

    def media(x):
        x = x[x > -1]
        return float(x.mean()) if x.size else float('nan')

    salida = {'map_torchmetrics': float(ev.stats[0])}
    for m, k in enumerate(LIMITES):
        salida[f'mAP50_95_max{k}'] = media(prec[:, :, 0, 0, m])
        salida[f'mAP50_max{k}'] = media(prec[i50, :, 0, 0, m])
        salida[f'mAP75_max{k}'] = media(prec[i75, :, 0, 0, m])
        salida[f'AR_max{k}'] = media(rec[:, 0, 0, m])
    return salida


def resumen_conteo(conteos):
    d = np.asarray(conteos) - tabla_img['anotadas'].to_numpy()
    return {'count_mae': float(np.abs(d).mean()), 'count_bias': float(d.mean())}


# Comprobación: un detector perfecto (las propias anotaciones como detecciones)
perfecto = [{'image_id': a['image_id'], 'category_id': 1, 'bbox': a['bbox'], 'score': 0.9} for a in gt_anns]
ref_perfecto = evaluar_coco(perfecto)
print('Detector perfecto:', {k: round(v, 3) for k, v in ref_perfecto.items() if k.startswith('mAP50_95') or k == 'map_torchmetrics'})
RESULTADOS = {}
CONTEOS = {}

## 3. YOLO11s y YOLO26s (misma inferencia que en NB08)

In [ ]:
import gc
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), 'No se detectó GPU. Seleccione Entorno de ejecución > Cambiar tipo de entorno > GPU.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))


def evaluar_yolo(nombre, ruta_modelo):
    yolo = YOLO(str(ruta_modelo))
    dets, conteos = [], [0] * len(imagenes)
    vistos = set()
    inicio = time.perf_counter()
    salidas = yolo.predict(source=str(LOCAL / 'images' / 'val'), imgsz=960, conf=0.001, iou=0.7, max_det=1000,
                           batch=1, stream=True, device=0, verbose=False)
    for salida in salidas:
        idx = INDICE[Path(salida.path).name]
        vistos.add(idx)
        if salida.boxes is None or len(salida.boxes) == 0:
            continue
        cajas = salida.boxes.xyxy.detach().cpu().numpy().astype(float)
        punt = salida.boxes.conf.detach().cpu().numpy().astype(float)
        conteos[idx] = int((punt >= UMBRAL_CONTEO).sum())
        for (x1, y1, x2, y2), s in zip(cajas, punt):
            dets.append({'image_id': idx, 'category_id': 1, 'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': float(s)})
    assert len(vistos) == len(imagenes), f'{nombre}: se procesaron {len(vistos)} de {len(imagenes)} imágenes'
    ms = (time.perf_counter() - inicio) * 1000 / len(imagenes)
    fila = {'modelo': nombre, **evaluar_coco(dets), **resumen_conteo(conteos), 'ms_por_imagen': ms,
            'detecciones_por_imagen_media': len(dets) / len(imagenes)}
    del yolo, salidas
    gc.collect(); torch.cuda.empty_cache()
    return fila, conteos


for nombre in ('YOLO11s', 'YOLO26s'):
    if not MODELOS[nombre].exists():
        print('Se omite', nombre, '(modelo no encontrado)'); continue
    RESULTADOS[nombre], CONTEOS[nombre] = evaluar_yolo(nombre, MODELOS[nombre])
    print(nombre, {k: round(v, 4) for k, v in RESULTADOS[nombre].items() if isinstance(v, float)})

## 4. Validación propia de Ultralytics

Es la evaluación que produjo el 0,756 del NB02. Se ejecuta con dos límites de detecciones por imagen: 300, que es el valor por defecto de Ultralytics, y 1 000.

In [ ]:
import yaml
yaml_local = LOCAL / 'dataset_val.yaml'
yaml_local.write_text(yaml.safe_dump({'path': str(LOCAL), 'train': 'images/val', 'val': 'images/val',
                                      'nc': 1, 'names': ['Pollo']}, sort_keys=False, allow_unicode=True), encoding='utf-8')
ULTRA = {}
for nombre in ('YOLO11s', 'YOLO26s'):
    if not MODELOS[nombre].exists():
        continue
    for limite in (300, 1000):
        modelo = YOLO(str(MODELOS[nombre]))
        m = modelo.val(data=str(yaml_local), split='val', imgsz=960, batch=8, conf=0.001, iou=0.7, max_det=limite,
                       device=0, plots=False, verbose=False)
        ULTRA[f'{nombre}_max{limite}'] = {'precision': float(m.box.mp), 'recall': float(m.box.mr),
                                          'mAP50': float(m.box.map50), 'mAP75': float(m.box.map75),
                                          'mAP50_95': float(m.box.map)}
        print(f'{nombre} · max_det = {limite}:', {k: round(v, 4) for k, v in ULTRA[f'{nombre}_max{limite}'].items()})
        del modelo
        gc.collect(); torch.cuda.empty_cache()

## 5. Faster R-CNN (misma arquitectura y pesos que en NB08)

In [ ]:
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF


def construir_faster_rcnn():
    modelo = fasterrcnn_resnet50_fpn_v2(
        weights=None, weights_backbone=None, min_size=640, max_size=960, box_detections_per_img=1000,
        rpn_pre_nms_top_n_train=3000, rpn_post_nms_top_n_train=2000,
        rpn_pre_nms_top_n_test=3000, rpn_post_nms_top_n_test=1500,
    )
    entrada = modelo.roi_heads.box_predictor.cls_score.in_features
    modelo.roi_heads.box_predictor = FastRCNNPredictor(entrada, 2)
    return modelo


if MODELOS['Faster R-CNN'].exists():
    frcnn = construir_faster_rcnn().to(DEVICE)
    frcnn.load_state_dict(torch.load(MODELOS['Faster R-CNN'], map_location=DEVICE, weights_only=True))
    frcnn.eval()
    dets, conteos = [], [0] * len(imagenes)
    inicio = time.perf_counter()
    with torch.inference_mode():
        for idx, ruta in enumerate(imagenes):
            imagen = TF.to_tensor(Image.open(ruta).convert('RGB')).to(DEVICE)
            pred = frcnn([imagen])[0]
            cajas = pred['boxes'].detach().cpu().numpy().astype(float)
            punt = pred['scores'].detach().cpu().numpy().astype(float)
            conteos[idx] = int((punt >= UMBRAL_CONTEO).sum())
            for (x1, y1, x2, y2), s in zip(cajas, punt):
                dets.append({'image_id': idx, 'category_id': 1, 'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': float(s)})
            if (idx + 1) % 100 == 0:
                print(f'  Faster R-CNN: {idx + 1}/{len(imagenes)}')
    ms = (time.perf_counter() - inicio) * 1000 / len(imagenes)
    RESULTADOS['Faster R-CNN'] = {'modelo': 'Faster R-CNN', **evaluar_coco(dets), **resumen_conteo(conteos), 'ms_por_imagen': ms,
                                  'detecciones_por_imagen_media': len(dets) / len(imagenes)}
    CONTEOS['Faster R-CNN'] = conteos
    print('Faster R-CNN', {k: round(v, 4) for k, v in RESULTADOS['Faster R-CNN'].items() if isinstance(v, float)})
    del frcnn
    gc.collect(); torch.cuda.empty_cache()
else:
    print('Se omite Faster R-CNN: no se encontró', MODELOS['Faster R-CNN'])

## 6. Resumen y descarga

In [ ]:
NB08 = {'YOLO11s': {'mAP50_95': 0.3998, 'mAP50': 0.9750, 'mAP75': 0.8620, 'count_mae': 13.867},
        'YOLO26s': {'mAP50_95': 0.4007, 'mAP50': 0.9772, 'mAP75': 0.8632, 'count_mae': 14.624},
        'Faster R-CNN': {'mAP50_95': 0.3314, 'mAP50': 0.8261, 'mAP75': 0.6549, 'count_mae': 41.611}}
filas = []
for nombre, r in RESULTADOS.items():
    filas.append({'Modelo': nombre, 'mAP50-95 en NB08': NB08[nombre]['mAP50_95'],
                  'Reproducido (máx. 100)': r['map_torchmetrics'], 'mAP50-95 (máx. 300)': r['mAP50_95_max300'],
                  'mAP50-95 (máx. 1000)': r['mAP50_95_max1000'], 'mAP50 (máx. 1000)': r['mAP50_max1000'],
                  'mAP75 (máx. 1000)': r['mAP75_max1000'], 'EAM de conteo': r['count_mae'], 'Sesgo de conteo': r['count_bias']})
resumen = pd.DataFrame(filas)
print(resumen.round(4).to_string(index=False))
print('\nValidación de Ultralytics:'); print(pd.DataFrame(ULTRA).T.round(4).to_string())

por_imagen = tabla_img.copy()
for nombre, c in CONTEOS.items():
    por_imagen['conteo_' + nombre.replace(' ', '_').replace('-', '')] = c
ruta_csv = DIR_VAL / 'conteo_detectores_por_imagen.csv'
por_imagen.to_csv(ruta_csv, index=False)
col_conteo = [c for c in por_imagen.columns if c.startswith('conteo_')]
eam_semana = {int(s): {c: round(float((g[c] - g['anotadas']).abs().mean()), 1) for c in col_conteo} | {'imagenes': int(len(g))}
              for s, g in por_imagen.groupby('semana')}
eam_galpon = {str(s): {c: round(float((g[c] - g['anotadas']).abs().mean()), 1) for c in col_conteo} | {'imagenes': int(len(g))}
              for s, g in por_imagen.groupby('galpon')}
print('\nEAM de conteo por semana:'); print(pd.DataFrame(eam_semana).T.to_string())

salida = {'n_imagenes': len(imagenes), 'aves_anotadas': int(n_aves.sum()),
          'imagenes_con_mas_de_100_aves': int((n_aves > 100).sum()), 'imagenes_con_mas_de_300_aves': int((n_aves > 300).sum()),
          'fraccion_de_aves_dentro_del_limite': techo, 'detector_perfecto': ref_perfecto,
          'parametros_inferencia': {'imgsz': 960, 'conf': 0.001, 'iou': 0.7, 'max_det': 1000, 'umbral_conteo': UMBRAL_CONTEO},
          'modelos': RESULTADOS, 'valores_NB08': NB08, 'validacion_ultralytics': ULTRA,
          'eam_conteo_por_semana': eam_semana, 'eam_conteo_por_galpon': eam_galpon,
          'entorno': {'torch': torch.__version__, 'torchvision': torchvision.__version__, 'gpu': torch.cuda.get_device_name(0)}}
ruta_json = DIR_VAL / 'reevaluacion_mAP.json'
ruta_json.write_text(json.dumps(salida, ensure_ascii=False, indent=2), encoding='utf-8')

ruta_zip = Path('/content/AgroVision_reevaluacion_mAP.zip') if EN_COLAB else DIR_VAL / 'AgroVision_reevaluacion_mAP.zip'
with zipfile.ZipFile(ruta_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(ruta_json, ruta_json.name); z.write(ruta_csv, ruta_csv.name)
print('\nGuardado en', DIR_VAL)
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_zip))